# Copy selection: what choosing duplicated BUSCO copies by peer similarity changes

BUSCO calls a gene *Duplicated* when a genome has several copies. A tree needs one sequence per genome and gene.
The old rule kept the copy with the **top BUSCO score**, but the best-scoring copy is not necessarily the orthologous one: a paralog
can score higher. The new rule (`multi_copy_selection: peer`, `scripts/peer_copy_selection.py`) aligns every candidate copy to the
**Complete** (single-copy) sequences of the same gene from genomes of the same taxon (the deepest level from genus up to phylum with
at least 3 peers; at most 8 peers used) and keeps the copy with the highest mean identity. Genomes without usable peers fall back to the
top score.

This notebook reads the decision log `multicopy_selection.tsv` that the aggregation writes (one row for every genome x gene with several
copies). It answers, in order:

1. **How many choices were made, and on what evidence?** (section 1)
2. **How often does the new rule overrule the old one, and is the new copy really more peer-like?** (section 2) - the share of
   choices that switch is exactly how often the old rule picked a copy other than the most peer-like one.
3. **Where does it happen?** Genes, lineages and genomes (section 3). A genome that switches for most of its genes is more likely
   a data problem (contamination, a doubled assembly) than an interesting biological case.
4. **Which choices should not be trusted?** (section 4)

The *Key findings* just below are generated from this run's numbers (computed at the end, moved up); each section has a one-line
**Takeaway** under its figure. What counts as OK / worth a look / a problem is set by the thresholds in section 0.

*Terms.* **switched** = the peer rule chose a different copy from the top-score rule. **gain** = mean identity (to the same peers) of
the chosen copy minus that of the top-score copy; a large gain means the old rule would have put a likely paralog into the alignment.

## 0. Setup

In [ ]:
import json, os, sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import copy_selection_lib as lib
import notebook_narrative as nn

OUT = Path(PARAMS["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)
print("Run:", PARAMS["label"])

frame, missing = lib.load_all(PARAMS["selections"])
if missing:
    print("No multicopy_selection.tsv for:", ", ".join(missing),
          "\n(re-run the aggregation with multi_copy_selection: peer to create it)")
if frame is None:
    raise SystemExit("No selection logs available; nothing to show.")

taxonomy = {}
if Path(PARAMS["taxonomy"]).exists():
    tax = pl.read_csv(PARAMS["taxonomy"], separator="\t", has_header=False, new_columns=["g", "t"], infer_schema_length=0, quote_char=None)
    taxonomy = dict(zip(tax["g"], tax["t"]))
frame = lib.attach_phylum(frame, taxonomy)
SOURCES = frame["source"].unique(maintain_order=True).to_list()
COLOURS = dict(zip(SOURCES, plt.cm.tab10.colors))


def show_all(df):
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1); cfg.set_tbl_rows(60); cfg.set_tbl_width_chars(200)
        print(df)


def finish(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", dpi=130)
    plt.show()

### Thresholds

What counts as a problem. Each rule has a *look* value (worth a look) and a *bad* value (a problem); change them here and every
verdict below follows.

In [ ]:
BY_PEERS = nn.Rule(look=0.90, bad=0.70, higher_is_worse=False)   # share of choices backed by peers (the rest fell back to the top score)
GENE_SWITCH = 0.50                                  # a gene is "often overruled" above this switch rate
GENE_SHARE = nn.Rule(look=0.05, bad=0.20)           # share of genes that are often overruled
GENOME_SWITCH = 0.60                                # a genome is suspicious above this switch rate
GENOME_SHARE = nn.Rule(look=0.02, bad=0.10)         # share of genomes that are suspicious
BIG_GAIN = 0.10                                     # a switch with at least this identity gain clearly matters
WEAK = nn.Rule(look=0.10, bad=0.25)                 # share of peer-decided choices that are weak (thin peers or no real ortholog)

FINDINGS = nn.Findings()
if missing:
    FINDINGS.add(nn.LOOK, "No decision log for: " + ", ".join(missing) + ".", evidence="multicopy_selection.tsv not found",
                 action="Re-run that aggregation with multi_copy_selection: peer.")

## 1. How many choices, and how many decided by peers?

Every row is one genome x gene with several candidate copies. *by_peers* are decisions backed by Complete copies from
the same taxon; the rest fell back to the top score because no peers exist (new lineages, or genes where nobody else
has a Complete copy). *share_switched* is among the peer-decided ones: how often the answer changed.

In [ ]:
overview = lib.overview(frame)
show_all(overview)
overview.write_csv(OUT / "overview.tsv", separator="\t")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
cand = lib.by_candidates(frame)
axes[0].bar(cand["n_candidates"].cast(pl.Utf8), cand["share_switched"], color="#4c78a8")
for i, n in enumerate(cand["decisions"]):
    axes[0].text(i, 0.01, f"n={n}", ha="center", fontsize=8, color="white")
axes[0].set_xlabel("candidate copies in the genome"); axes[0].set_ylabel("share switched")
axes[0].set_title("More copies, more room to pick a paralog")
lev = lib.by_peer_level(frame)
axes[1].barh(lev["level"], lev["decisions"], color="#72b7b2")
axes[1].set_xscale("log"); axes[1].set_xlabel("decisions (log)")
axes[1].set_title("Taxonomic level the peers came from")
finish(fig, "decisions")
show_all(lev.select("level", "decisions", "share_switched", "mean_peers"))

by_peers_share = {r["source"]: r["share_by_peers"] for r in overview.iter_rows(named=True)}
sw_share = {r["source"]: r["share_switched"] for r in overview.iter_rows(named=True)}
s_p, t_p = nn.across(BY_PEERS, by_peers_share, nn.pct)
_, t_s = nn.across(nn.Rule(1, 2), sw_share, nn.pct)
total = int(overview["decisions"].sum())
FINDINGS.say(s_p, f"{total:,} genome x gene choices had several copies; {t_p} were decided by peers (the rest fell back to the top BUSCO score). "
                  f"Among those, the peer rule chose a different copy than the top score in {t_s}.",
             evidence=f"decided by peers {t_p}",
             action="A low peer share means few Complete copies exist for these taxa; add related genomes or accept the top-score fallback.")

## 2. How much better is the chosen copy?

Each point is a switched choice: the identity of the copy the old rule would have used (x) against the copy now used (y),
both measured against the same peers. Points far above the diagonal are the clear wins: the top-scoring copy was much less
similar to its peers than another copy, i.e. probably a paralog. Points close to the diagonal are near-ties where the choice
hardly matters for the tree.

In [ ]:
g = lib.gains(frame)
print(f"{g.height} switched choices; median gain {g['gain'].median():.3f}, "
      f"{(g['gain'] > 0.1).sum()} with gain > 0.1, {(g['gain'] > 0.25).sum()} with gain > 0.25")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for source in SOURCES:
    s = g.filter(pl.col("source") == source)
    axes[0].scatter(s["top_score_identity"], s["chosen_identity"], s=5, alpha=0.3, color=COLOURS[source], label=source)
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[0].set_xlabel("identity to peers: top-score copy (old rule)"); axes[0].set_ylabel("identity to peers: chosen copy (new rule)")
axes[0].legend(markerscale=3, fontsize=8)
bins = np.linspace(0, max(0.5, float(g["gain"].max() or 0.5)), 40)
for source in SOURCES:
    axes[1].hist(g.filter(pl.col("source") == source)["gain"], bins=bins, alpha=0.6, color=COLOURS[source], label=source)
axes[1].set_yscale("log"); axes[1].set_xlabel("identity gain of the chosen copy"); axes[1].set_ylabel("switched choices (log)")
axes[1].set_title("Most switches are small, a tail is large")
finish(fig, "gain")

big = int((g["gain"] >= BIG_GAIN).sum())
n_sw = g.height
FINDINGS.say(nn.OK,
             f"{n_sw:,} choices switched; the median identity gain is {g['gain'].median():.3f}, and {big:,} ({nn.pct(big / max(n_sw, 1))}) gain at least {BIG_GAIN:.2f}. "
             f"The small switches are near-ties; the large ones are copies that scored best but resemble their peers much less than another copy, i.e. likely paralogs "
             f"that the old rule would have put into the alignment.",
             evidence=f"{big} switches with gain >= {BIG_GAIN}")

## 3. Where does the rule disagree with the top score?

* *Genes*: a gene with a high switch rate usually has a frequent paralog (BUSCO's top hit often is the paralog).
* *Phyla*: lineages with ancient genome duplications or many segmental duplications switch more.
* *Genomes*: a genome that switches for most of its multicopy genes is suspicious (contamination or a doubled assembly) rather
  than interesting; look at it before trusting its sequences in the tree.

In [ ]:
genes = lib.gene_table(frame)
phyla = lib.phylum_table(frame)
genomes = lib.genome_table(frame)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(genes["share_switched"], bins=30, color="#4c78a8"); axes[0].set_xlabel("share of decisions switched"); axes[0].set_ylabel("genes")
axes[0].set_title("Per gene (>= 20 decisions)")
top = phyla.head(15)
axes[1].barh(top["phylum"][::-1], top["share_switched"][::-1], color="#f58518"); axes[1].set_xlabel("share switched")
axes[1].set_title("Per phylum (>= 50 decisions)")
axes[2].hist(genomes["share_switched"], bins=30, color="#54a24b"); axes[2].set_xlabel("share of the genome's multicopy genes switched"); axes[2].set_ylabel("genomes")
axes[2].set_title("Per genome (>= 10 decisions)")
finish(fig, "where")
print("Genes switched most often:"); show_all(genes.head(10))
print("Genomes switched most often (check these first):"); show_all(genomes.head(10))
genes.write_csv(OUT / "genes.tsv", separator="\t"); genomes.write_csv(OUT / "genomes.tsv", separator="\t"); phyla.write_csv(OUT / "phyla.tsv", separator="\t")

gene_share = float((genes["share_switched"] >= GENE_SWITCH).mean()) if genes.height else 0.0
genome_share = float((genomes["share_switched"] >= GENOME_SWITCH).mean()) if genomes.height else 0.0
often = int((genes["share_switched"] >= GENE_SWITCH).sum())
suspicious = int((genomes["share_switched"] >= GENOME_SWITCH).sum())
FINDINGS.say(GENE_SHARE.status(gene_share),
             f"{often} of {genes.height} genes ({nn.pct(gene_share)}) are overruled in at least {nn.pct(GENE_SWITCH)} of their decisions"
             + (": " + ", ".join(f"`{g_}`" for g_ in genes.head(3)["busco_id"]) + " lead." if often else "."),
             evidence=f"{often} genes", action="Genes with a frequent paralog are where the top BUSCO score is least reliable; consider checking their gene trees.")
FINDINGS.say(GENOME_SHARE.status(genome_share),
             f"{suspicious} of {genomes.height} genomes ({nn.pct(genome_share, 1)}) switch for at least {nn.pct(GENOME_SWITCH)} of their multicopy genes"
             + (": " + ", ".join(f"`{g_}`" for g_ in genomes.head(3)["genome_acc"]) + " lead." if suspicious else "."),
             evidence=f"{suspicious} genomes",
             action="Check these genomes' assemblies for duplication or contamination before trusting their sequences in the tree.")
if phyla.height:
    top_phylum = phyla.row(0, named=True)
    nn.show(f"The phylum with the highest switch rate is **{top_phylum['phylum']}** ({nn.pct(top_phylum['share_switched'])} of {top_phylum['decisions']:,} decisions); "
            "lineages with ancient whole-genome duplications are expected to lead.")

## 4. How much should the choices be trusted?

A choice is weak when the peer set is thin (fewer than 3 peers), or when even the best candidate resembles its peers poorly
(identity below 0.4: probably no true ortholog among the candidates, e.g. a lost or badly predicted copy). Weak choices are
few if the table below is dominated by the first row. Switches with a tiny gain are harmless either way.

In [ ]:
weak = lib.weak_choices(frame)
weak_df = pl.DataFrame({"check": list(weak), "choices": list(weak.values())}).with_columns(
    (pl.col("choices") / weak["decided by peers"]).alias("share_of_peer_decisions"))
show_all(weak_df)
weak_df.write_csv(OUT / "weak_choices.tsv", separator="\t")
frame.write_csv(OUT / "all_decisions.tsv", separator="\t")

decided = weak["decided by peers"]
weak_total = int(frame.filter(pl.col("decided_by_peers") & ((pl.col("n_peers") < 3) | (pl.col("chosen_identity") < 0.4))).height)
weak_share_value = weak_total / max(decided, 1)
FINDINGS.say(WEAK.status(weak_share_value),
             f"{weak_total:,} peer-decided choices ({nn.pct(weak_share_value, 1)}) are weak: {weak['fewer than 3 peers']:,} rest on fewer than 3 peers and "
             f"{weak['best copy identity < 0.4']:,} have a best copy that resembles its peers by less than 0.4.",
             evidence=f"{weak_total} weak choices",
             action="Weak choices are the ones to revisit if a specific placement looks wrong; they are listed in all_decisions.tsv.")

## 5. Summary and what to do next

The numbers to take away are in the *Key findings* above and in sections 1 and 2: the share of choices that switched (how often the
old rule picked another copy) and the gain of those switches (how much more peer-like the new copy is). A small share with small
gains means the tree barely changes; a larger share with sizeable gains means paralogs were entering the alignment and are now replaced.

Below, every finding that was not fine, worst first.

To measure the effect on the tree itself, build the `multi` tree twice (`multi_copy_selection: peer` and `top_score`, in the metapackage
and tree configs) and compare them in `tree_accuracy.ipynb` (fragmentation, misplaced tips, RF distance).

In [ ]:
nn.show("### What to do next\n\n" + FINDINGS.actions())
pl.DataFrame(FINDINGS.table()).write_csv(OUT / "findings.tsv", separator="\t")

## Key findings

In [ ]:
nn.show(FINDINGS.verdict() + "\n\n" + FINDINGS.bullets() +
        "\n\n*Generated from this run's numbers; the thresholds are in section 0. Details and figures follow.*")